## Preprocessing the datasets of SH cars (OLX, AutoVit, BestAuto, Mobile.de) ##
Pipeline complet de integrare, curățare, normalizare, deduplicare și centralizare a datelor auto din 4 surse majore.


In [1]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


In [2]:
# Definire cai relative catre cele 4 seturi de date
# Functioneaza indiferent daca notebook-ul este rulat din extraction/ sau din radacina repo-ului
repo_root = Path("..") if (Path("..") / "data").exists() else Path(".")
data_dir = repo_root / "data"

dataset_paths = {
    "OLX": data_dir / "olx/cars_with_brands.csv",
    "AutoVit": data_dir / "autovit/autovit_prices_cleand.csv",
    "BestAuto": data_dir / "bestauto/bestauto.csv",
    "MobileDe": data_dir / "mobilede/mobile_de_ro.csv"
}

# Incarcare dataset-uri
datasets = {}
for name, path in dataset_paths.items():
    print(f"Incarcare {name} din {path}...")
    datasets[name] = pd.read_csv(path, low_memory=False)
    print(f"  -> {len(datasets[name])} randuri, {len(datasets[name].columns)} coloane")

# Normalizare nume coloane (Column mapping)
column_mapping = {
    "OLX": {
        "URL": "url",
        "Title": "title",
        "Listing price": "price",
        "Model": "model",
        "Capacitate motor": "engine_capacity",
        "Caroserie": "body_type",
        "Culoare": "color",
        "An de fabricatie": "year",
        "Numar de usi": "doors",
        "Stare": "condition",
        "Combustibil": "fuel_type",
        "Cutie de viteze": "transmission",
        "Rulaj": "mileage",
        "Inmatriculat/Neimatriculat": "registered",
        "Putere": "power",
        "brand": "brand"
    },

    "AutoVit": {
        "url": "url",
        "title": "title",
        "price": "price",
        "listing_mileage": "mileage",
        "listing_fuel_type": "fuel_type",
        "Anul productiei": "year",
        "listing_year": "year_listing",
        "Marca": "brand",
        "Model": "model",
        "Culoare": "color",
        "Numar de portiere": "doors",
        "Capacitate cilindrica": "engine_capacity",
        "Putere": "power",
        "body_type": "body_type",
        "Cutie de viteze": "transmission",
        "Stare": "condition",
        "Inmatriculat": "registered"
    },

    "BestAuto": {
        "listing_url": "url",
        "title": "title",
        "price": "price",
        "brand": "brand",
        "model": "model",
        "first_registration": "year",
        "mileage_km": "mileage",
        "engine_displacement_cm3": "engine_capacity",
        "power_hp": "power",
        "fuel_type": "fuel_type",
        "body_type": "body_type",
        "transmission": "transmission",
        "doors": "doors",
        "color": "color",
        "condition": "condition"
    },

    "MobileDe": {
        "listing_url": "url",
        "title": "title",
        "price": "price",
        "brand": "brand",
        "model": "model",
        "first_registration": "year",
        "mileage_km": "mileage",
        "engine_displacement_cm3": "engine_capacity",
        "power_hp": "power",
        "fuel_type": "fuel_type",
        "body_type": "body_type",
        "transmission": "transmission",
        "doors": "doors",
        "color": "color_detail",
        "color_base": "color",
        "condition": "condition"
    }
}


Incarcare OLX din olivian part\OLX\olx_car_dataset\cars_with_brands.csv...
  -> 26065 randuri, 25 coloane
Incarcare AutoVit din olivian part\AutoVit\autovit_prices_cleand.csv...
  -> 18454 randuri, 35 coloane
Incarcare BestAuto din dan part\bestauto\bestauto.csv...
  -> 24144 randuri, 28 coloane
Incarcare MobileDe din dan part\mobilede\mobile_de_ro.csv...
  -> 19992 randuri, 32 coloane


In [3]:
# Aplicare column mapping si armonizare campuri initiale
prepared = {}

for name, data in datasets.items():
    prepared[name] = data.rename(
        columns=column_mapping[name]
    ).copy()

# Armonizare status inmatriculare pentru BestAuto si MobileDe
prepared["BestAuto"]["registered"] = prepared["BestAuto"]["condition"].map({
    "inmatriculata": "Inmatriculat",
    "inmatriculabila": "Neinmatriculat",
    "leasing": "Inmatriculat"
})
prepared["BestAuto"]["condition"] = "Utilizat"

# Armonizare an AutoVit: 'Anul productiei' este cel exact din fisa tehnica; daca lipseste, completam cu listing_year
if "year_listing" in prepared["AutoVit"].columns:
    prepared["AutoVit"]["year"] = prepared["AutoVit"]["year"].combine_first(
        prepared["AutoVit"]["year_listing"]
    )

# Extragere an pentru MobileDe (din formatul MM/YYYY) si setare status inmatriculare
prepared["MobileDe"]["year"] = pd.to_numeric(
    prepared["MobileDe"]["year"].astype(str).str.extract(r"(\d{4})")[0],
    errors="coerce"
)
prepared["MobileDe"]["registered"] = "Neinmatriculat"

# Verificare coloane duplicate
for name, data in prepared.items():
    duplicates = data.columns[data.columns.duplicated()].tolist()
    print(f"{name}:")
    print("  Randuri:", len(data))
    print("  Coloane:", len(data.columns))
    print("  Coloane duplicate:", duplicates)
    print()


OLX:
  Randuri: 26065
  Coloane: 25
  Coloane duplicate: []

AutoVit:
  Randuri: 18454
  Coloane: 35
  Coloane duplicate: []

BestAuto:
  Randuri: 24144
  Coloane: 29
  Coloane duplicate: []

MobileDe:
  Randuri: 19992
  Coloane: 33
  Coloane duplicate: []



In [4]:
# Identificare coloane comune tuturor celor 4 dataset-uri
common_columns = sorted(
    set.intersection(*[set(data.columns) for data in prepared.values()])
)

print("Coloane comune identificate:", common_columns)
print(f"Total coloane comune: {len(common_columns)}")

# Combinare dataset-uri folosind doar coloanele comune + coloana 'source'
df = pd.concat(
    [
        data[common_columns].assign(source=name)
        for name, data in prepared.items()
    ],
    ignore_index=True
)

print("\nDataset unificat:")
print("Total randuri:", len(df))
print("Total coloane:", len(df.columns))
print("\nDistributie dupa sursa:")
print(df["source"].value_counts().to_string())


Coloane comune identificate: ['body_type', 'brand', 'color', 'condition', 'doors', 'engine_capacity', 'fuel_type', 'mileage', 'model', 'power', 'price', 'registered', 'title', 'transmission', 'url', 'year']
Total coloane comune: 16

Dataset unificat:
Total randuri: 88655
Total coloane: 17

Distributie dupa sursa:
source
OLX         26065
BestAuto    24144
MobileDe    19992
AutoVit     18454


In [5]:
# Normalizare coloane de identificare
for col in ["brand", "model"]:
    df[col] = df[col].astype("string").str.lower().str.strip()

# 1. Eliminare duplicate dupa URL
initial_count = len(df)
df = df.drop_duplicates(
    subset=["url"],
    keep="first"
)
print(f"Randuri eliminate dupa URL: {initial_count - len(df)}")

# 2. Deduplicare stricta pe specificatii tehnice ale masinii
duplicate_columns = [
    "brand",
    "model",
    "year",
    "engine_capacity",
    "fuel_type",
    "power",
    "transmission",
    "mileage",
    "body_type",
    "doors"
]

valid = df.dropna(subset=duplicate_columns).copy()
invalid = df[df[duplicate_columns].isna().any(axis=1)].copy()

# Pastrarea inregistrarii cu cele mai multe campuri completate
valid["_completeness"] = valid.notna().sum(axis=1)
valid = valid.sort_values("_completeness", ascending=False)

valid = valid.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
).drop(columns="_completeness")

df = pd.concat([valid, invalid], ignore_index=True).reset_index(drop=True)

print(f"Randuri ramase dupa deduplicare completa: {len(df)}")


Randuri eliminate dupa URL: 50
Randuri ramase dupa deduplicare completa: 87647


# Data Quality Inspection & Cleaning #


In [6]:
# ============================================================
# DATA QUALITY INSPECTION
# ============================================================

for col in df.columns:
    print("\n" + "=" * 80)
    print(f"COLUMN: {col}")
    print("Data type:", df[col].dtype)
    print("Total values:", len(df))
    print("Missing values:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())

    print("\nMOST COMMON VALUES:")
    print(df[col].value_counts(dropna=False).head(15).to_string())

    print("\nRANDOM EXAMPLES:")
    print(
        df[col]
        .dropna()
        .astype(str)
        .sample(min(10, df[col].notna().sum()), random_state=42)
        .to_list()
    )

    values = df[col].dropna().astype(str)
    checks = {
        "Leading/trailing spaces": values.str.contains(r"^\s|\s$", regex=True),
        "Multiple consecutive spaces": values.str.contains(r"\s{2,}", regex=True),
        "Contains digits": values.str.contains(r"\d", regex=True),
    }

    print("\nPOTENTIAL FORMATTING ISSUES:")
    for issue, mask in checks.items():
        print(f"  {issue}: {mask.sum()}")



COLUMN: body_type
Data type: str
Total values: 87647
Missing values: 421
Unique values: 27

MOST COMMON VALUES:
body_type
Berlina            22977
SUV                16358
Hatchback          10566
Combi               8438
Suv                 5459
Break               5236
Sedan               3952
Monovolum           3147
Coupé               1782
Compacta            1717
Coupe               1626
Cabrio              1601
Altele              1042
Utilitara            849
Masina de teren      575

RANDOM EXAMPLES:
['Hatchback', 'Hatchback', 'Coupé', 'Hatchback', 'Hatchback', 'Combi', 'Berlina', 'Berlina', 'Berlina', 'Combi']

POTENTIAL FORMATTING ISSUES:
  Leading/trailing spaces: 0
  Multiple consecutive spaces: 0
  Contains digits: 0

COLUMN: brand
Data type: string
Total values: 87647
Missing values: 101
Unique values: 112

MOST COMMON VALUES:
brand
volkswagen       14608
bmw              12163
mercedes-benz    11416
audi              8050
ford              4728
renault           4261
d

In [7]:
# ============================================================
# DATA CLEANING AND VALUE CORRECTION
# ============================================================

# 1. YEAR
# Corectare valori ca 20210 -> 2021, 19950 -> 1995
df["year"] = pd.to_numeric(df["year"].astype(str).str.extract(r"(\d{4,5})")[0], errors="coerce")
mask_yr = df["year"] > 10000
df.loc[mask_yr, "year"] = df.loc[mask_yr, "year"] / 10
df["year"] = df["year"].round().astype("Int64")

# 2. PRICE
# Curatare format preturi OLX (eliminare €, spatii, virgula ca zecimala)
df["price"] = pd.to_numeric(
    df["price"].astype(str)
    .str.replace("€", "", regex=False)
    .str.replace(r"\s+", "", regex=True)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# 3. MILEAGE
# Extragere valoare numerica corecta (fara eliminarea punctului zecimal din float)
df["mileage"] = pd.to_numeric(
    df["mileage"].astype(str)
    .str.replace("km", "", case=False, regex=False)
    .str.replace(r"\s+", "", regex=True),
    errors="coerce"
).round().astype("Int64")

# 4. ENGINE CAPACITY
# Curatare si corectare valori AutoVit formatate eronat (19973 -> 1997, 196830 -> 1968)
df["engine_capacity"] = pd.to_numeric(
    df["engine_capacity"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)
mask_ec = df["engine_capacity"] > 10000
values_ec = df.loc[mask_ec, "engine_capacity"].astype("float64")
corrected_ec = values_ec.where(values_ec % 100 != 30, values_ec / 100)
corrected_ec = corrected_ec.where(values_ec % 100 == 30, values_ec / 10)
df.loc[mask_ec, "engine_capacity"] = corrected_ec.round()

# Corectare valori introduse in litri (ex: 1.6 sau 2.0 L -> 1600/2000 cm3; 16 sau 20 -> 1600/2000 cm3)
mask_l1 = (df["engine_capacity"] >= 0.6) & (df["engine_capacity"] < 8)
df.loc[mask_l1, "engine_capacity"] = (df.loc[mask_l1, "engine_capacity"] * 1000).round()
mask_l2 = (df["engine_capacity"] >= 8) & (df["engine_capacity"] < 100)
df.loc[mask_l2, "engine_capacity"] = (df.loc[mask_l2, "engine_capacity"] * 100).round()
df["engine_capacity"] = df["engine_capacity"].round().astype("Int64")

# 5. POWER
df["power"] = pd.to_numeric(
    df["power"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
).round().astype("Int64")

# 6. DOORS
# Normalizare '4/5' -> 5, '2/3' -> 3, '50.0' -> 5, '40.0' -> 4
def clean_doors(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s == "4/5": return 5
    if s == "2/3": return 3
    if s == "6/7": return 6
    num = pd.to_numeric(re.sub(r"[^\d.]", "", s), errors="coerce")
    if pd.isna(num): return np.nan
    if num > 6: num = num / 10
    return round(num)

df["doors"] = df["doors"].apply(clean_doors).astype("Int64")

# 7. BODY TYPE
df["body_type"] = df["body_type"].replace({
    "Berlina": "Sedan",
    "Sedan": "Sedan",
    "Break": "Combi",
    "Combi": "Combi",
    "Compacta": "Compact",
    "Masina de oras": "Compact",
    "Mini": "Compact",
    "Compact": "Compact",
    "Hatchback": "Hatchback",
    "Hatchback, Vehicul de prezentare": "Hatchback",
    "Hatchback, Inmatriculare de o zi": "Hatchback",
    "Suv": "SUV",
    "SUV": "SUV",
    "SUV / Teren / Pickup": "SUV",
    "Masina de teren": "SUV",
    "Off-road": "SUV",
    "Coupé": "Coupe",
    "Coupe": "Coupe",
    "Sportwagen/Coupé": "Coupe",
    "Sport": "Coupe",
    "Masina sport / Coupé": "Coupe",
    "Cabrio": "Cabrio",
    "Decapotabila": "Cabrio",
    "Roadster": "Cabrio",
    "Monovolum": "Monovolum",
    "Van/Minibus": "Monovolum",
    "Van / Minibus": "Monovolum",
    "Microbus": "Monovolum",
    "Utilitara": "Utilitara",
    "Pickup": "Utilitara",
    "Altele": "Altele"
})

# 8. COLOR
df["color"] = df["color"].replace({
    "negru": "Negru", "Negru": "Negru",
    "gri": "Gri", "Gri": "Gri",
    "alb": "Alb", "Alb": "Alb",
    "albastru": "Albastru", "Albastru": "Albastru",
    "argintiu": "Argintiu", "Argintiu": "Argintiu", "Argint": "Argintiu",
    "rosu": "Roșu", "Rosu": "Roșu", "roșu": "Roșu", "Roşu": "Roșu",
    "maro": "Maro", "Maro": "Maro",
    "verde": "Verde", "Verde": "Verde",
    "bej": "Bej", "Bej": "Bej",
    "galben": "Galben/Auriu", "Galben": "Galben/Auriu", "Galben / Auriu": "Galben/Auriu",
    "Galben/Auriu": "Galben/Auriu", "auriu": "Galben/Auriu", "Auriu": "Galben/Auriu",
    "portocaliu": "Portocaliu", "Portocaliu": "Portocaliu",
    "visiniu": "Vișiniu", "Visiniu": "Vișiniu",
    "violet": "Violet", "Violet": "Violet",
    "Alta culoare": "Alta culoare"
})

# 9. CONDITION
df["condition"] = df["condition"].replace({
    "USED": "Utilizat",
    "inmatriculata": "Utilizat",
    "inmatriculabila": "Utilizat",
    "leasing": "Utilizat",
    "Utilizat, Fara accident": "Utilizat",
    "Utilizat, Dauna reparata": "Utilizat",
    "Utilizat, Fara accident, Neutilizabil": "Utilizat",
    "Utilizat, Neutilizabil": "Utilizat",
    "Utilizat, Dauna reparata, Neutilizabil": "Utilizat",
    "Utilizat, Avariat, Fara accident": "Utilizat",
    "Nou": "Nou"
})

# 10. REGISTRATION
df["registered"] = df["registered"].replace({
    "Da": "Inmatriculat",
    "TRUE": "Inmatriculat",
    "True": "Inmatriculat",
    "Nu": "Neinmatriculat",
    "FALSE": "Neinmatriculat",
    "False": "Neinmatriculat"
})

# 11. FUEL TYPE
df["fuel_type"] = df["fuel_type"].replace({
    "Diesel": "Diesel",
    "Diesel (compatibil biodiesel)": "Diesel",
    "Benzina": "Benzină",
    "Benzină": "Benzină",
    "Benzina (compatibil E10)": "Benzină",
    "Benzina (compatibil biodiesel, E10)": "Benzină",
    "GPL": "Benzină + GPL",
    "Benzina + GPL": "Benzină + GPL",
    "Benzina + gpl": "Benzină + GPL",
    "Benzina+gaz": "Benzină + GPL",
    "Gaz": "Benzină + GPL",
    "Benzina + cng": "Benzină + GPL",
    "Hibrid": "Hibrid",
    "Plug-in Hybrid": "Plug-in hybrid",
    "Hibrid - plug-in": "Plug-in hybrid",
    "Hibrid plug-in (benzina/electric)": "Plug-in hybrid",
    "Plug-in hybrid": "Plug-in hybrid",
    "Electric": "Electric",
    "Electrica": "Electric",
    "Electrică": "Electric"
})

# 12. TRANSMISSION
df["transmission"] = df["transmission"].replace({
    "Automata": "Automată",
    "automata": "Automată",
    "Automată": "Automată",
    "Manuala": "Manuală",
    "manuala": "Manuală",
    "Manuală": "Manuală"
})

# 13. REMOVE UNNECESSARY WHITESPACE
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Curatarea si normalizarea valorilor a fost finalizata cu succes!")


Curatarea si normalizarea valorilor a fost finalizata cu succes!


In [8]:
# ============================================================
# VALIDATION AFTER CORRECTIONS
# ============================================================

for col in [
    "year",
    "engine_capacity",
    "doors",
    "body_type",
    "color",
    "condition",
    "registered",
    "fuel_type",
    "transmission"
]:
    print("\n" + "=" * 60)
    print(col.upper())
    print(df[col].value_counts(dropna=False).head(20).to_string())

# Inspect suspicious numerical values
print("\nSUSPICIOUS NUMERICAL VALUES")

for col, minimum, maximum in [
    ("year", 1950, 2026),
    ("engine_capacity", 400, 10000),
    ("doors", 2, 6),
    ("power", 20, 2000),
    ("mileage", 0, 1500000),
    ("price", 100, 2000000)
]:
    suspicious = df[
        ~df[col].between(minimum, maximum) & df[col].notna()
    ][col]

    print(f"\n{col}: {len(suspicious)} suspicious values")
    print(suspicious.value_counts().head(10).to_string())



YEAR
year
2019    6095
2018    6054
2017    5924
2016    5911
2015    5259
2012    4864
2014    4797
2020    4776
2021    4772
2011    4448
2013    4294
2022    4131
2010    3592
2023    3087
2009    3056
2008    2987
2007    2769
2006    2115
2024    1494
2005    1483

ENGINE_CAPACITY
engine_capacity
2993    9011
1968    6198
2000    5759
1998    5236
1995    3897
1598    3659
1600    3575
<NA>    2190
1500    1797
999     1735
1461    1535
1400    1513
1498    1416
1390    1393
1200    1348
1197    1275
3000    1135
1499    1133
2967    1094
1997     976

DOORS
doors
5       53503
<NA>    18741
4        9616
3        4422
2        1274
6          91

BODY_TYPE
body_type
Sedan        26929
SUV          23116
Combi        13674
Hatchback    10568
Monovolum     3602
Coupe         3584
Cabrio        1837
Compact       1806
Utilitara     1068
Altele        1042
NaN            421

COLOR
color
NaN             31270
Negru           14933
Gri             13420
Alb              9639
Albastru

In [9]:
# Inspect suspicious numerical values with their corresponding records

checks = {
    "year": df["year"] > 2026,
    "engine_capacity": df["engine_capacity"].notna() & (df["engine_capacity"] < 400),
    "power": df["power"].notna() & (df["power"] < 20),
    "mileage": df["mileage"] > 1_500_000,
    "price": df["price"].notna() & (df["price"] < 100)
}

for col, mask in checks.items():
    print("\n" + "=" * 70)
    print(f"{col.upper()} — {mask.sum()} records")

    columns = [
        c for c in [
            "brand", "model", "year", "engine_capacity",
            "power", "mileage", "price", "title", "url", "source"
        ]
        if c in df.columns
    ]

    print(df.loc[mask, columns].head(15).to_string(index=False))



YEAR — 8 records
     brand       model  year  engine_capacity  power  mileage        price                                                 title                                                                                                              url  source
volkswagen        golf  2027             1498    130        1 2.684400e+04                                       Volkswagen Golf                                           https://www.autovit.ro/autoturisme/anunt/volkswagen-golf-ID7HQT6L.html AutoVit
volkswagen        golf  2027             1498    115        1 1.870000e+04                                       Volkswagen Golf                                           https://www.autovit.ro/autoturisme/anunt/volkswagen-golf-ID7HQT6C.html AutoVit
land rover range rover  2027             2997    350       10 1.675850e+05 Land Rover Range Rover 3.0 I6 D350 MHEV Autobiography https://www.autovit.ro/autoturisme/anunt/land-rover-range-rover-ver-3-0-i6-d350-mhev-autobiography-ID7H

# Outlier Filtering, Centralized Dataset Export & Visualizations #


In [10]:
# ============================================================
# OUTLIER FILTERING & EXPORT CENTRALIZED DATASET
# ============================================================

# Filtram inregistrarile cu valori evidente de tip eroare (outliers extremi)
valid_mask = (
    (df["year"].isna() | df["year"].between(1950, 2026)) &
    (df["price"].isna() | df["price"].between(100, 2_000_000)) &
    (df["mileage"].isna() | (df["mileage"] <= 1_500_000)) &
    (df["engine_capacity"].isna() | df["engine_capacity"].between(400, 10_000)) &
    (df["power"].isna() | df["power"].between(20, 2_000)) &
    (df["doors"].isna() | df["doors"].between(2, 6))
)

df_clean = df[valid_mask].copy().reset_index(drop=True)

output_dir = data_dir / "centralized"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "cars_centralized.csv"
df_clean.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Salvare fisier centralizat: {output_file}")
print(f"Initial: {len(df)} randuri -> Dupa filtrare erori: {len(df_clean)} randuri")
print("\nDistributia finala pe surse:")
print(df_clean["source"].value_counts().to_string())
print("\nInformatii despre dataset-ul final:")
print(df_clean.info())


Salvare fisier centralizat: cars_centralized.csv
Initial: 87647 randuri -> Dupa filtrare erori: 85391 randuri

Distributia finala pe surse:
source
OLX         25837
BestAuto    23420
MobileDe    19384
AutoVit     16750

Informatii despre dataset-ul final:
<class 'pandas.DataFrame'>
RangeIndex: 85391 entries, 0 to 85390
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   body_type        84970 non-null  str    
 1   brand            85290 non-null  string 
 2   color            54625 non-null  str    
 3   condition        82504 non-null  str    
 4   doors            67190 non-null  Int64  
 5   engine_capacity  83260 non-null  Int64  
 6   fuel_type        85124 non-null  str    
 7   mileage          84728 non-null  Int64  
 8   model            85103 non-null  string 
 9   power            82542 non-null  Int64  
 10  price            84886 non-null  float64
 11  registered       73389 non-null  str    
 1

In [11]:
# ============================================================
# VIZUALIZARI SI ANALIZA COMPARATIVA PE DATASET-UL CENTRALIZAT
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distributia dupa sursa
df_clean["source"].value_counts().plot(kind="bar", ax=axes[0, 0], color="skyblue", edgecolor="black")
axes[0, 0].set_title("Numar de anunturi per Sursa")
axes[0, 0].set_ylabel("Numar anunturi")
axes[0, 0].grid(axis="y", linestyle="--", alpha=0.7)

# 2. Distributia anilor de fabricatie (1990 - 2026)
years = df_clean["year"].dropna()
years = years[(years >= 1990) & (years <= 2026)]
axes[0, 1].hist(years, bins=36, color="salmon", edgecolor="black")
axes[0, 1].set_title("Distributia Anilor de Fabricatie (1990-2026)")
axes[0, 1].set_xlabel("An")
axes[0, 1].grid(axis="y", linestyle="--", alpha=0.7)

# 3. Top 10 Marci Auto
df_clean["brand"].value_counts().head(10).plot(kind="barh", ax=axes[1, 0], color="lightgreen", edgecolor="black")
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Top 10 Marci Auto")
axes[1, 0].set_xlabel("Numar anunturi")
axes[1, 0].grid(axis="x", linestyle="--", alpha=0.7)

# 4. Distributia Combustibilului
df_clean["fuel_type"].value_counts().plot(kind="pie", ax=axes[1, 1], autopct="%1.1f%%", startangle=140)
axes[1, 1].set_ylabel("")
axes[1, 1].set_title("Ponderea Tipurilor de Combustibil")

plt.tight_layout()
plot_file = output_dir / "distributii_dataset_centralizat.png"
plt.savefig(plot_file, dpi=150)
plt.close()
print(f"Grafice generate si salvate in '{plot_file}'!")


Grafice generate si salvate in 'distributii_dataset_centralizat.png'!
